In [ ]:
import pandas as pd

# Extract the base document URL and append /export?format=csv
file_id = "156vASY2oHddtQbZIYPWwLyPcm1bVIp7Z"
url = f"https://docs.google.com/spreadsheets/d/{file_id}/export?format=csv"

df = pd.read_csv(url)

print(df.shape)
display(df.head())

(29, 79)


,No,Komoditas (Rp),01/ 06/ 2026,02/ 06/ 2026,03/ 06/ 2026,04/ 06/ 2026,05/ 06/ 2026,08/ 06/ 2026,09/ 06/ 2026,10/ 06/ 2026,...,02/ 09/ 2026,03/ 09/ 2026,04/ 09/ 2026,07/ 09/ 2026,08/ 09/ 2026,09/ 09/ 2026,10/ 09/ 2026,11/ 09/ 2026,14/ 09/ 2026,15/ 09/ 2026
0,I,Beras,"14,550","14,550","14,550","14,550","14,550","14,550","14,550","14,550",...,"14,850","14,850","14,850","14,850","14,850",-,"14,850","14,850",-,"14,750"
1,1,Beras Kualitas Medium I,"14,650","14,650","14,650","14,650","14,650","14,650","14,650","14,650",...,"15,000","15,000","15,000","15,000","15,000","15,000","15,000","15,000","15,000","15,000"
2,2,Beras Kualitas Medium II,"13,000","13,000","13,000","13,000","13,000","13,000","13,000","13,000",...,"13,400","13,400","13,400","13,400","13,400","13,400","13,400","13,400","13,400","13,400"
3,3,Beras Kualitas Super I,"15,250","15,250","15,250","15,250","15,250","15,250","15,250","15,250",...,"15,450","15,450","15,450","15,450","15,450","15,500","15,500","15,500","15,400","15,250"
4,4,Beras Kualitas Super II,"15,150","15,150","15,150","15,150","15,150","15,150","15,150","15,150",...,"15,500","15,500","15,500","15,500","15,500","15,500","15,500","15,500","15,400","15,250"


In [ ]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

In [ ]:

df_ml = df.iloc[:, 1:].copy()

# Nama kolom komoditas
df_ml = df_ml.rename(columns={
    df_ml.columns[0]: "komoditas"
})

df_long = df_ml.melt(
    id_vars=["komoditas"],
    var_name="tanggal",
    value_name="harga"
)

print(df_long.shape)
display(df_long.head())

(2233, 3)


,komoditas,tanggal,harga
0,Beras,01/ 06/ 2026,"14,550"
1,Beras Kualitas Medium I,01/ 06/ 2026,"14,650"
2,Beras Kualitas Medium II,01/ 06/ 2026,"13,000"
3,Beras Kualitas Super I,01/ 06/ 2026,"15,250"
4,Beras Kualitas Super II,01/ 06/ 2026,"15,150"


In [ ]:
df_long["harga"] = (
    df_long["harga"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.replace(".", "", regex=False)
)

df_long["harga"] = pd.to_numeric(
    df_long["harga"],
    errors="coerce"
)

df_long["tanggal"] = pd.to_datetime(
    df_long["tanggal"],
    dayfirst=True,
    errors="coerce"
)

df_long = df_long.dropna(
    subset=["komoditas", "tanggal", "harga"]
)

df_long = df_long.sort_values(
    ["komoditas", "tanggal"]
)

print(df_long.shape)

(2213, 3)


In [ ]:
# One-hot encoding untuk nama komoditas
df_ml_final = pd.get_dummies(
    df_long,
    columns=["komoditas"],
    dtype=int
)

# Pisahkan fitur dan target
X = df_ml_final.drop(
    columns=["harga", "tanggal"]
)

y = df_ml_final["harga"]

# Simpan urutan fitur agar prediksi berikutnya konsisten
feature_columns = X.columns.tolist()

print("Jumlah data:", len(X))
print("Jumlah fitur:", len(feature_columns))
display(X.head())

Jumlah data: 2010
Jumlah fitur: 38


,hari,bulan,tahun,hari_dalam_minggu,harga_h1,harga_h2,harga_h3,harga_h7,rata2_7hari,komoditas_Bawang Merah,...,komoditas_Daging Sapi Kualitas 2,komoditas_Gula Pasir,komoditas_Gula Pasir Kualitas Premium,komoditas_Gula Pasir Lokal,komoditas_Minyak Goreng,komoditas_Minyak Goreng Curah,komoditas_Minyak Goreng Kemasan Bermerk 1,komoditas_Minyak Goreng Kemasan Bermerk 2,komoditas_Telur Ayam,komoditas_Telur Ayam Ras Segar
215,10,6,2026,2,52250.0,52750.0,55250.0,51500.0,53571.428571,1,...,0,0,0,0,0,0,0,0,0,0
244,11,6,2026,3,52250.0,52250.0,52750.0,53250.0,53678.571429,1,...,0,0,0,0,0,0,0,0,0,0
273,12,6,2026,4,52250.0,52250.0,52250.0,54750.0,53535.714286,1,...,0,0,0,0,0,0,0,0,0,0
302,15,6,2026,0,49500.0,52250.0,52250.0,55250.0,52785.714286,1,...,0,0,0,0,0,0,0,0,0,0
331,16,6,2026,1,45250.0,49500.0,52250.0,55250.0,51357.142857,1,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
# Ambil tanggal unik secara berurutan
tanggal_unik = sorted(
    df_ml_final["tanggal"].unique()
)

# Gunakan 80% tanggal awal untuk training
batas = int(len(tanggal_unik) * 0.8)

if batas == 0 or batas >= len(tanggal_unik):
    raise ValueError(
        "Data tanggal tidak cukup untuk pembagian training dan testing."
    )

tanggal_batas = tanggal_unik[batas]

train_mask = (
    df_ml_final["tanggal"] < tanggal_batas
)

test_mask = (
    df_ml_final["tanggal"] >= tanggal_batas
)

X_train = X.loc[train_mask]
X_test = X.loc[test_mask]

y_train = y.loc[train_mask]
y_test = y.loc[test_mask]

print("Tanggal awal training:",
      df_ml_final.loc[train_mask, "tanggal"].min())

print("Tanggal akhir training:",
      df_ml_final.loc[train_mask, "tanggal"].max())

print("Tanggal awal testing:",
      df_ml_final.loc[test_mask, "tanggal"].min())

print("Jumlah data training:", len(X_train))
print("Jumlah data testing:", len(X_test))

Tanggal awal training: 2026-06-10 00:00:00
Tanggal akhir training: 2026-08-26 00:00:00
Tanggal awal testing: 2026-08-27 00:00:00
Jumlah data training: 1624
Jumlah data testing: 386


In [ ]:
from sklearn.ensemble import RandomForestRegressor

model_rf = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    max_depth=None
)

model_rf.fit(X_train, y_train)

print("Training Random Forest selesai!")

Training Random Forest selesai!


In [ ]:
import numpy as np
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Prediksi Random Forest
y_pred = model_rf.predict(X_test)

# Baseline: prediksi harga sama dengan observasi sebelumnya
y_baseline = df_ml_final.loc[
    test_mask, "harga_h1"
].to_numpy()

# Evaluasi Random Forest
mae_rf = mean_absolute_error(y_test, y_pred)

rmse_rf = np.sqrt(
    mean_squared_error(y_test, y_pred)
)

r2_rf = r2_score(y_test, y_pred)

# Evaluasi baseline
mae_baseline = mean_absolute_error(
    y_test, y_baseline
)

rmse_baseline = np.sqrt(
    mean_squared_error(y_test, y_baseline)
)

print("=== RANDOM FOREST ===")
print(f"MAE  : Rp {mae_rf:,.2f}")
print(f"RMSE : Rp {rmse_rf:,.2f}")
print(f"R²   : {r2_rf:.4f}")

print("\n=== BASELINE HARGA TERAKHIR ===")
print(f"MAE  : Rp {mae_baseline:,.2f}")
print(f"RMSE : Rp {rmse_baseline:,.2f}")

if mae_rf < mae_baseline:
    print("\nRandom Forest lebih baik berdasarkan MAE.")
else:
    print(
        "\nRandom Forest belum mengungguli baseline "
        "berdasarkan MAE."
    )

=== RANDOM FOREST ===
MAE  : Rp 666.85
RMSE : Rp 1,614.76
R²   : 0.9981

=== BASELINE HARGA TERAKHIR ===
MAE  : Rp 293.65
RMSE : Rp 787.11

Random Forest belum mengungguli baseline berdasarkan MAE.


In [ ]:
hasil = df_ml_final.loc[
    test_mask,
    ["tanggal", "harga", "harga_h1"]
].copy()

hasil["Prediksi_RF"] = y_pred
hasil["Prediksi_Baseline"] = y_baseline

hasil["Error_RF"] = (
    hasil["harga"] - hasil["Prediksi_RF"]
).abs()

hasil["Error_Baseline"] = (
    hasil["harga"] - hasil["Prediksi_Baseline"]
).abs()

hasil = hasil.rename(
    columns={
        "tanggal": "Tanggal",
        "harga": "Harga_Aktual"
    }
)

display(hasil.head(20))

,Tanggal,Harga_Aktual,harga_h1,Prediksi_RF,Prediksi_Baseline,Error_RF,Error_Baseline
1839,2026-08-27,27750.0,27750.0,27593.333333,27750.0,156.666667,0.0
1868,2026-08-28,27250.0,27750.0,27588.333333,27750.0,338.333333,500.0
1897,2026-08-31,26250.0,27250.0,27113.500000,27250.0,863.500000,1000.0
1926,2026-09-01,26250.0,26250.0,26241.833333,26250.0,8.166667,0.0
1955,2026-09-02,26250.0,26250.0,26788.333333,26250.0,538.333333,0.0
1984,2026-09-03,26250.0,26250.0,26605.833333,26250.0,355.833333,0.0
2013,2026-09-04,26250.0,26250.0,26150.833333,26250.0,99.166667,0.0
2042,2026-09-07,26750.0,26250.0,27053.666667,26250.0,303.666667,500.0
2071,2026-09-08,26750.0,26750.0,26927.166667,26750.0,177.166667,0.0
2129,2026-09-10,26500.0,26750.0,26326.833333,26750.0,173.166667,250.0


In [ ]:
# Latih ulang model menggunakan seluruh data yang tersedia
# setelah evaluasi selesai
model_rf_final = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    max_depth=None
)

model_rf_final.fit(X, y)

# Membuat prediksi untuk setiap komoditas
hasil_prediksi = []

for komoditas in sorted(df_long["komoditas"].unique()):

    histori = df_long[
        df_long["komoditas"] == komoditas
    ].sort_values("tanggal")

    histori = histori.drop_duplicates(
        subset=["tanggal"],
        keep="last"
    )

    harga = histori["harga"].to_numpy()

    if len(harga) < 7:
        print(
            f"Data tidak cukup untuk {komoditas}, dilewati."
        )
        continue

    # Tanggal satu hari setelah observasi terakhir
    tanggal_terakhir = histori["tanggal"].max()

    tanggal_prediksi = (
        tanggal_terakhir + pd.Timedelta(days=1)
    )

    # Bentuk fitur yang sama dengan saat training
    input_baru = pd.DataFrame([{
        "hari": tanggal_prediksi.day,
        "bulan": tanggal_prediksi.month,
        "tahun": tanggal_prediksi.year,
        "hari_dalam_minggu": tanggal_prediksi.dayofweek,
        "harga_h1": harga[-1],
        "harga_h2": harga[-2],
        "harga_h3": harga[-3],
        "harga_h7": harga[-7],
        "rata2_7hari": np.mean(harga[-7:]),
        "komoditas": komoditas
    }])

    # One-hot encoding dan samakan kolom dengan training
    input_baru = pd.get_dummies(
        input_baru,
        columns=["komoditas"],
        dtype=int
    )

    input_baru = input_baru.reindex(
        columns=feature_columns,
        fill_value=0
    )

    # Prediksi
    prediksi = model_rf_final.predict(
        input_baru
    )[0]

    hasil_prediksi.append({
        "Komoditas": komoditas,
        "Tanggal_Observasi_Terakhir": tanggal_terakhir,
        "Tanggal_Prediksi": tanggal_prediksi,
        "Harga_Terakhir": harga[-1],
        "Prediksi_Harga": round(prediksi, 2)
    })

df_hasil_prediksi = pd.DataFrame(
    hasil_prediksi
)

display(df_hasil_prediksi)

,Komoditas,Tanggal_Observasi_Terakhir,Tanggal_Prediksi,Harga_Terakhir,Prediksi_Harga
0,Bawang Merah,2026-09-15,2026-09-16,25500.0,25550.50
1,Bawang Merah Ukuran Sedang,2026-09-15,2026-09-16,25500.0,25847.17
2,Bawang Putih,2026-09-15,2026-09-16,28500.0,28543.33
3,Bawang Putih Ukuran Sedang,2026-09-15,2026-09-16,28500.0,28585.00
4,Beras,2026-09-15,2026-09-16,14750.0,14800.50
5,Beras Kualitas Medium I,2026-09-15,2026-09-16,15000.0,15000.00
6,Beras Kualitas Medium II,2026-09-15,2026-09-16,13400.0,13400.00
7,Beras Kualitas Super I,2026-09-15,2026-09-16,15250.0,15311.33
8,Beras Kualitas Super II,2026-09-15,2026-09-16,15250.0,15308.17
9,Cabai Merah,2026-09-15,2026-09-16,33500.0,33895.83


In [ ]:
!pip install firebase-admin

In [2]:
import firebase_admin
from firebase_admin import credentials

cred = credentials.Certificate("pasarku-3d6f3-firebase-adminsdk-fbsvc-c8474a38d4.json")
firebase_admin.initialize_app(cred)


In [7]:
import firebase_admin
from firebase_admin import credentials, firestore

# Inisialisasi (jika belum diinisialisasi)
if not firebase_admin._apps:
    cred = credentials.Certificate("pasarku-3d6f3-firebase-adminsdk-fsvc-c8474a38d4.json")
    firebase_admin.initialize_app(cred)

# Definisikan objek db (Firestore Client)
db = firestore.client()
print("Koneksi Firestore (db) berhasil disiapkan!")

Koneksi Firestore (db) berhasil disiapkan!


In [14]:
import time

# Fungsi callback yang otomatis dipicu saat ada data baru/perubahan di Firestore
def on_snapshot(col_snapshot, changes, read_time):
    print(f"\n[REAL-TIME] Terdeteksi perubahan data pada {read_time}!")

    # Panggil fungsi prediksi dan simpan otomatis yang sudah dibuat sebelumnya
    try:
        prediksi_dan_simpan_ke_firebase()
    except Exception as e:
        print(f"Gagal melakukan prediksi otomatis: {e}")

# Daftarkan listener ke koleksi 'data_pasarku'
query_watch = db.collection('data_pasarku')
doc_watch = query_watch.on_snapshot(on_snapshot)

print("Listener Real-Time Aktif! Menunggu data masuk di Firebase...")
print("Jangan tutup atau hentikan cell ini jika ingin terus memantau secara real-time.")

# Loop agar cell tetap aktif mendengarkan perubahan di Colab
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    # Untuk menghentikan listener, Anda bisa menekan tombol stop (pause) di Google Colab
    doc_watch.unsubscribe()
    print("\nListener real-time dihentikan.")

Listener Real-Time Aktif! Menunggu data masuk di Firebase...
Jangan tutup atau hentikan cell ini jika ingin terus memantau secara real-time.

[REAL-TIME] Terdeteksi perubahan data pada 2026-10-27 13:06:56.866136+00:00!
Prediksi berhasil dijalankan dan hasil tersimpan di Firebase Firestore!

Listener real-time dihentikan.


In [13]:
import pandas as pd
import numpy as np

def prediksi_dan_simpan_ke_firebase():
    # Pastikan library pandas dan numpy terbaca di dalam fungsi
    global pd, np, db, model_rf_final, feature_columns

    # 1. Ambil seluruh data dari koleksi 'data_pasarku' di Firestore
    docs = db.collection('data_pasarku').stream()
    data_list = []

    for doc in docs:
        d = doc.to_dict()
        data_list.append({
            'komoditas': d.get('komoditas'),
            'tanggal': pd.to_datetime(d.get('tanggal')),
            'harga': float(d.get('harga', 0))
        })

    if not data_list:
        print("Tidak ada data ditemukan di Firestore.")
        return

    df_firestore = pd.DataFrame(data_list)
    df_firestore = df_firestore.sort_values(['komoditas', 'tanggal'])

    hasil_prediksi_baru = []

    # 2. Lakukan prediksi untuk setiap komoditas
    for komoditas in df_firestore['komoditas'].unique():
        histori = df_firestore[df_firestore['komoditas'] == komoditas].sort_values('tanggal')
        histori = histori.drop_duplicates(subset=["tanggal"], keep="last")
        harga = histori['harga'].to_numpy()

        if len(harga) < 7:
            continue  # Data kurang dari 7 hari, lewati

        tanggal_terakhir = histori['tanggal'].max()
        tanggal_prediksi = tanggal_terakhir + pd.Timedelta(days=1)

        # Bentuk fitur input sesuai format training model Random Forest
        input_baru = pd.DataFrame([{
            "hari": tanggal_prediksi.day,
            "bulan": tanggal_prediksi.month,
            "tahun": tanggal_prediksi.year,
            "hari_dalam_minggu": tanggal_prediksi.dayofweek,
            "harga_h1": harga[-1],
            "harga_h2": harga[-2],
            "harga_h3": harga[-3],
            "harga_h7": harga[-7],
            "rata2_7hari": np.mean(harga[-7:]),
            "komoditas": komoditas
        }])

        # One-hot encoding & samakan kolom dengan feature_columns model training
        input_baru = pd.get_dummies(input_baru, columns=["komoditas"], dtype=int)
        input_baru = input_baru.reindex(columns=feature_columns, fill_value=0)

        # Prediksi harga
        prediksi = model_rf_final.predict(input_baru)[0]

        data_hasil = {
            "komoditas": komoditas,
            "tanggal_observasi_terakhir": str(tanggal_terakhir.date()),
            "tanggal_prediksi": str(tanggal_prediksi.date()),
            "harga_terakhir": float(harga[-1]),
            "prediksi_harga": round(float(prediksi), 2)
        }

        hasil_prediksi_baru.append(data_hasil)

        # 3. Simpan hasil prediksi secara otomatis kembali ke Firestore (koleksi 'hasil_prediksi')
        doc_id = f"{komoditas.replace(' ', '_')}_{tanggal_prediksi.date()}"
        db.collection('hasil_prediksi').document(doc_id).set(data_hasil)

    print("Prediksi berhasil dijalankan dan hasil tersimpan di Firebase Firestore!")
    return pd.DataFrame(hasil_prediksi_baru)